# Centripetal force — step-by-step simulation (Colab)
**Physics, Year 3 — Liceo «L. da Vinci», Trento**

**VPython does not work** in Google Colab (Colab uses its own notebook system, which is not compatible with VPython's 3D graphics). Here we run **the same simulation** with the step-by-step method and draw the results with `matplotlib`.

What we simulate: a ball of mass $m$ starts with speed $v_0$; a force of constant magnitude $F$ acts on it, **always perpendicular to the velocity**. At each step $\Delta t$:
$$a = \frac{F}{m},\qquad v_{n+1} = v_n + a\,\Delta t,\qquad x_{n+1} = x_n + v_{n+1}\,\Delta t$$

Run the cells one at a time (▶ on the left, or Shift+Enter).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate(m, F, v0, dt, duration, t_cut=None):
    """Step-by-step method. Returns times, x and y positions, speed.
    If t_cut is a number, from that instant the force disappears (the string is cut)."""
    x, y = 0.0, 0.0           # starts at the origin...
    vx, vy = v0, 0.0          # ...moving to the right
    T, X, Y, V = [0.0], [x], [y], [v0]
    t = 0.0
    while t < duration:
        v = np.hypot(vx, vy)
        if t_cut is not None and t >= t_cut:
            ax, ay = 0.0, 0.0                       # no force
        else:
            px, py = -vy / v, vx / v                # unit vector perpendicular to v (to the left)
            ax, ay = F / m * px, F / m * py         # Newton's second law
        vx, vy = vx + ax * dt, vy + ay * dt         # new velocity
        x, y = x + vx * dt, y + vy * dt             # new position
        t += dt
        T.append(t); X.append(x); Y.append(y); V.append(np.hypot(vx, vy))
    return np.array(T), np.array(X), np.array(Y), np.array(V)

def measure(T, X, Y):
    """Radius and period measured on the first lap."""
    for i in range(1, len(X)):
        if X[i-1] < 0 <= X[i] and T[i] > 0.1:          # back at the starting point
            return Y[:i+1].max() / 2, T[i]
    return Y.max() / 2, None

## 1. What path do we get? Does the speed change?
Before running, write down your prediction. Then change the parameters and run again.

In [ ]:
m, F, v0 = 0.50, 2.0, 2.0     # kg, N, m/s
dt, duration = 0.001, 3.5      # s

T, X, Y, V = simulate(m, F, v0, dt, duration)
r, Tp = measure(T, X, Y)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.5))
a1.plot(X, Y); a1.plot(0, 0, 'o'); a1.set_aspect('equal'); a1.grid(alpha=.3)
a1.set_title('Path'); a1.set_xlabel('x (m)'); a1.set_ylabel('y (m)')
a2.plot(T, V); a2.set_ylim(0, 1.5 * V.max()); a2.grid(alpha=.3)
a2.set_title('Speed'); a2.set_xlabel('t (s)'); a2.set_ylabel('|v| (m/s)')
plt.show()
print(f"radius r = {r:.3f} m    period T = {Tp:.3f} s    F*r/(m*v0^2) = {F*r/(m*v0**2):.3f}")

## 2. The string is cut
At $t = 1.0\ \mathrm{s}$ the force disappears. Where does the ball go? Why?

In [ ]:
T, X, Y, V = simulate(0.50, 2.0, 2.0, 0.001, 3.0, t_cut=1.0)
plt.figure(figsize=(5.5, 5)); plt.plot(X, Y); plt.gca().set_aspect('equal'); plt.grid(alpha=.3)
plt.title('The string is cut at t = 1.0 s'); plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.show()

## 3. What does the radius depend on?
Keep $m$ and $F$ fixed and change $v_0$. Then try changing $F$ and $m$ yourself (one thing at a time).

In [ ]:
m, F = 0.50, 2.0
print(" v0 (m/s) |  r (m)  |  T (s)  | F*r/(m*v0^2)")
list_v, list_r = [], []
for v0 in [1.0, 2.0, 3.0, 4.0]:
    T, X, Y, V = simulate(m, F, v0, 0.001, 2.2 * np.pi * m * v0 / F)
    r, Tp = measure(T, X, Y)
    list_v.append(v0); list_r.append(r)
    print(f"   {v0:4.1f}   |  {r:5.3f}  |  {Tp:5.3f}  |   {F*r/(m*v0**2):5.3f}")

v2 = np.array(list_v)**2
plt.figure(figsize=(5.5, 4)); plt.plot(v2, list_r, 'o-'); plt.grid(alpha=.3)
plt.xlabel('$v_0^2$ (m$^2$/s$^2$)'); plt.ylabel('r (m)'); plt.title('The radius as a function of $v_0^2$'); plt.show()

**Questions.** If $v_0$ doubles, how does $r$ change? And if $F$ doubles? And if $m$ doubles? Write a formula linking $F$, $m$, $v_0$ and $r$.

## 4. Challenge: the time step
Set `dt = 0.1` in the cell of point 1 and run it again. What happens to the path and to $|v|$? Is it physics or an error of the method? How do you fix it?